In [ ]:
# This notebook will extract the snapshot of precipitation surrounding each cyclone separately, and
# save it as its own netcdf file. It will also output a summary array of all the cyclones, providing
# an index to look up cyclones with the highest total rainfall, the greatest rainfall field for each cyclone etc.

In [1]:
# data I/O and wrangling
import xarray as xr
import pandas as pd
import numpy as np
import glob
from datetime import datetime
import matplotlib.dates as dt


In [2]:
# Set the radius around the cyclone center to capture the rainfall
radius = 10 # degrees lat/lon from center of weather system

# Load the cyclone csv dataset
res = 'HiRes'
level = 900
dataset = 'ERA5'

precip_data_root = '/g/data/rt52/era5/single-levels/reanalysis/tp/'
cyclone_dir = '/g/data/gb02/nxg561/03_Thermal_Structure/July2026/Climatology/LowRes/ERA5/'
project_dir = '/g/data/gb02/nxg561/Inverse_Cyclone/'
save_path = project_dir + 'Figures/'

lats = (-25, -55)
lons = (100, 160)

years = (1979, 2018)
n_years = years[1] - years[0] + 1

In [3]:
# cyclone data I/O
        
# open the cyclone dataframe
cyclone_cps_filename = cyclone_dir + str(level) + 'hpa_tracks_saust_cps.csv'
cyclone_cps_df = pd.read_csv(cyclone_cps_filename, sep=',')
        
# convert the time column to datetime format
cyclone_cps_df["time"] = pd.to_datetime(cyclone_cps_df[["year", "month", "day", "hour"]])
        
# Retain only the cyclone centers in our study region
cyclone_cps_df = cyclone_cps_df[
    (cyclone_cps_df['lat'] < lats[0]) &
    (cyclone_cps_df['lat'] > lats[1]) &
    (cyclone_cps_df['lon'] > lons[0]) &
    (cyclone_cps_df['lon'] < lons[1])]
        
# Now take a subset of the cyclone dataframe for the four thermal classes
cold_cyclone_df = cyclone_cps_df[(cyclone_cps_df['VTL'] < 0) & (cyclone_cps_df['VTU'] < 0)]
hybrid_cyclone_df = cyclone_cps_df[(cyclone_cps_df['VTL'] > 0) & (cyclone_cps_df['VTU'] < 0)]
warm_cyclone_df = cyclone_cps_df[(cyclone_cps_df['VTL'] > 0) & (cyclone_cps_df['VTU'] > 0)]
inverse_cyclone_df = cyclone_cps_df[(cyclone_cps_df['VTL'] < 0) & (cyclone_cps_df['VTU'] > 0)]

In [4]:
master_output_dict = {}

# This part will be turned into a loop over all the years after testing a single year

for year in (1979, 1980):

    print(year)

    year_output_dict = {}
    
    for month in range(1,13):
    
        print(month)
        
        # Test the code on a smaller subset of the cyclone dataset
        cyclone_cps_df_month = cyclone_cps_df[(cyclone_cps_df['year']==year) & (cyclone_cps_df['month']==month)].copy()
    
        # ==============================================
        # Open all the rainfall data files for one year
        precip_data_path = precip_data_root + str(year) + '/'
        
        # glob the single file for that month
        file = glob.glob(precip_data_path + "tp_era5_oper_sfc_" + str(year) + str(month).zfill(2) + "*.nc")[0]  # match files starting with "prefix"
        
        precip_ds = xr.open_dataset(file, engine='netcdf4')
        
        # extract the precip dataframe from the dataset, and sum the data over 6 hour intervals
        rain = precip_ds['tp'].resample(time="6h", label="left").sum()
    
        #==============================================
        tp_max = []
        tp_mean = []
        
        for _, row in cyclone_cps_df_month.iterrows():
            
            # select time slice
            rain_t = rain.sel(time=row["time"])
            
            lat0 = row["lat"]
            lon0 = row["lon"]
            
            # compute distance
            distance = np.sqrt((rain_t.latitude - lat0)**2 + (rain_t.longitude - lon0)**2)
            
            # mask
            tp_subset = rain_t.where(distance <= radius)
            
            # store max
            tp_max.append(tp_subset.max().item())
            # store mean
            tp_mean.append(tp_subset.mean().item())
        
        # write the values of max tp and mean tp into the cyclone dataframe for this month and convert m to mm
        cyclone_cps_df_month["tp_max"] = tp_max
        cyclone_cps_df_month["tp_mean"] = tp_mean
    
        cyclone_cps_df_month["tp_max"] = cyclone_cps_df_month["tp_max"] * 1000
        cyclone_cps_df_month["tp_mean"] = cyclone_cps_df_month["tp_mean"] * 1000
    
        # gather the cyclone dataframe for this month in a dictionary, to be concatenated later. 
        year_output_dict[month] = cyclone_cps_df_month

    # concatenate the data for each month into a single dataframe for that year, and add that to the master dictionary
    print('Concatenating year_output_df')
    year_output_df = pd.concat(year_output_dict.values(), ignore_index=True)
    master_output_dict[year] = year_output_df
    print(type(master_output_dict[year]))

1979
1
2
3
4
5
6
7
8
9
10
11
12
Concatenating year_output_df
<class 'pandas.core.frame.DataFrame'>
1980
1
2
3
4
5
6
7
8
9
10
11
12
Concatenating year_output_df
<class 'pandas.core.frame.DataFrame'>


In [5]:
len(master_output_dict)

2

In [6]:
master_output_df = pd.concat(master_output_dict.values(), ignore_index=True)
master_output_df

,track_id,year,month,day,hour,time,lon,lat,test2,B,VTL,VTU,tp_max,tp_mean
0,9,1979,1,16,12,1979-01-16 12:00:00,135.0,-25.5,54.577968,7.993546,-12.955221,-6.928288,92.225357,2.636089
1,9,1979,1,16,18,1979-01-16 18:00:00,136.5,-25.5,60.029412,7.816965,3.183162,-0.103801,120.076537,3.026575
2,9,1979,1,17,0,1979-01-17 00:00:00,138.0,-25.5,57.928795,0.434385,12.879580,-20.442082,75.171504,2.407504
3,9,1979,1,17,12,1979-01-17 12:00:00,138.0,-25.5,50.001552,-8.545058,43.180767,-36.914926,71.290696,2.107125
4,9,1979,1,18,0,1979-01-18 00:00:00,138.0,-25.5,57.431835,6.911343,55.411350,17.106792,87.554345,1.596899
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2463,27899,1980,12,29,0,1980-12-29 00:00:00,108.0,-43.5,99.384867,50.217691,-111.737246,-361.178578,26.306537,2.027537
2464,27899,1980,12,29,6,1980-12-29 06:00:00,112.5,-45.0,117.036215,74.334366,-80.293553,-366.530692,14.905524,1.611206
2465,27899,1980,12,29,12,1980-12-29 12:00:00,117.0,-49.5,100.713847,47.906669,-101.259724,-269.994157,18.251115,1.706887
2466,27899,1980,12,29,18,1980-12-29 18:00:00,120.0,-51.0,104.656097,30.172669,-2.400563,-213.434099,10.262769,1.406493


In [7]:
master_output_df.to_csv(project_dir + str(level) + "hpa_tracks_saust_cps_precip.csv", index=False)

In [3]:
# Open the file from the entire cyclone dataset and isolate the inverse hybrid cyclones
cyclone_cps_filename = project_dir + '900hpa_tracks_saust_cps_precip.csv'
cyclone_cps_df = pd.read_csv(cyclone_cps_filename, sep=',')

# Now take a subset of the cyclone dataframe for the four thermal classes
cold_cyclone_df = cyclone_cps_df[(cyclone_cps_df['VTL'] < 0) & (cyclone_cps_df['VTU'] < 0)]
hybrid_cyclone_df = cyclone_cps_df[(cyclone_cps_df['VTL'] > 0) & (cyclone_cps_df['VTU'] < 0)]
warm_cyclone_df = cyclone_cps_df[(cyclone_cps_df['VTL'] > 0) & (cyclone_cps_df['VTU'] > 0)]
inverse_cyclone_df = cyclone_cps_df[(cyclone_cps_df['VTL'] < 0) & (cyclone_cps_df['VTU'] > 0)]

In [4]:
inverse_cyclone_df

,track_id,year,month,day,hour,time,lon,lat,test2,B,VTL,VTU,tp_max,tp_mean
6,9,1979,1,18,12,1979-01-18 12:00:00,138.0,-25.5,53.175235,-2.943570,-14.123315,27.934361,32.072835,0.540540
7,9,1979,1,18,18,1979-01-18 18:00:00,136.5,-25.5,51.593318,-1.401634,-35.922327,35.480148,27.971178,0.480548
8,9,1979,1,19,0,1979-01-19 00:00:00,136.5,-27.0,40.707244,-6.238622,-19.721543,40.232579,12.694327,0.341601
9,9,1979,1,19,6,1979-01-19 06:00:00,136.5,-27.0,36.214054,-11.026888,-9.110019,43.826085,16.319818,0.888390
10,9,1979,1,19,12,1979-01-19 12:00:00,136.5,-27.0,28.910473,15.718691,-34.947142,33.735161,25.200996,0.752846
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49102,524215,2018,2,20,18,2018-02-20 18:00:00,124.5,-25.5,64.598669,8.759374,-25.298399,43.973476,51.530179,1.722606
49235,526353,2018,4,8,12,2018-04-08 12:00:00,112.5,-54.0,103.258795,34.193311,-70.495118,22.901142,10.183566,1.361874
49464,528703,2018,6,15,0,2018-06-15 00:00:00,145.5,-48.0,64.263869,23.744732,-19.498234,30.051482,15.995203,1.399892
49656,530782,2018,8,19,6,2018-08-19 06:00:00,124.5,-52.5,122.590659,-15.297627,-20.759883,24.007016,9.072911,1.821576


In [12]:
np.max(inverse_cyclone_df['tp_max'])

156.0538568590277

In [14]:
# save the inverse hybrid cyclone dataframe to its own file
inverse_cyclone_df.to_csv(project_dir + '900hpa_tracks_saust_cps_precip_inverse.csv', index=False)

In [18]:
melb_lats = (-28, -48)
melb_lons = (135, 155)

# Retain only the cyclone centers in our study region
inverse_cyclone__melb_df = inverse_cyclone_df[
    (inverse_cyclone_df['lat'] < melb_lats[0]) &
    (inverse_cyclone_df['lat'] > melb_lats[1]) &
    (inverse_cyclone_df['lon'] > melb_lons[0]) &
    (inverse_cyclone_df['lon'] < melb_lons[1])]

# save the inverse hybrid cyclone dataframe to its own file
inverse_cyclone__melb_df.to_csv(project_dir + '900hpa_tracks_saust_cps_precip_inverse_melb.csv', index=False)

In [17]:
-25.5 < -28

False